In [ ]:
# In[1]:

import json

# Load the task.json file to understand the alert context
task_file_path = 'rca100/cases/t086/task.json'

with open(task_file_path, 'r') as file:
    task_data = json.load(file)

alert_title = task_data.get('alert_title')
alert_window = task_data.get('alert_window')
alert_entity = task_data.get('alert_entity')
prompt_text = task_data.get('prompt_text')

alert_title, alert_window, alert_entity, prompt_text

```
Out[1]:
```
The alert is about a sudden increase in response time for the operation `cart::POST /oteldemo.CartService/AddItem`. The alert time window is from `2026-04-04 23:29:47.051744` to `2026-04-04 23:35:04` (UTC+8). The user is requesting help to analyze the root cause of this issue.

The original code execution output of IPython Kernel is also provided below for reference:

('', {'start': '2026-04-04T23:29:47.051744+08:00', 'end': '2026-04-04T23:35:04+08:00'}, {'entity_id': 'e433b3f842dc4e525a834701cc603c28', 'entity_name': 'cart::POST /oteldemo.CartService/AddItem', 'entity_type': 'apm.operation', 'entity_domain': 'apm'}, '<code vibeops_object type="alert_event">cart响应时间突增告警<alert_event event_id="0dfccbf47fbd564c494a33353fbcd5dc" trans_id="uolj6kl98h06pe4d3njlf6ava1" rule_id="e8b430fd-f3db-4f75-bf3b-11ed13c00b33" rule_name="cart响应时间突增告警" /></code>\n\n帮我分析下根因。')```
```

In [ ]:
# In[2]:

import pandas as pd

# Load the metrics.parquet file
metrics_file_path = 'rca100/cases/t086/metrics.parquet'
metrics_df = pd.read_parquet(metrics_file_path)

# List all available KPIs
available_kpis = metrics_df['metric'].unique()
available_kpis

```
Out[2]:
```
The available KPIs in the metrics data include various node-level metrics (e.g., `node_cpu_usage_rate`, `node_memory_usage_rate`), deployment-level metrics (e.g., `deployment_cpu_usage_total`, `deployment_availability_rate`), cluster-level metrics (e.g., `cluster_node_count`, `cluster_failed_node_count`), namespace-level metrics, and application-level metrics such as `error_count`, `slow_count`, `request_count`, `latency`, `error_rate`, and JVM/thread pool metrics. These KPIs will help analyze performance issues related to resource usage, errors, and latency.

The original code execution output of IPython Kernel is also provided below for reference:

['node_ready_status' 'node_not_ready_status' 'node_memory_usage_rate'
 'node_pod_running_count' 'node_disk_usage_rate' 'node_cpu_usage_rate'
 'deployment_desired_replicas' 'deployment_available_replicas'
 'deployment_ready_replicas' 'deployment_availability_rate'
 'deployment_memory_usage_total' 'deployment_cpu_usage_vs_limits'
 'deployment_cpu_usage_vs_requests' 'deployment_cpu_usage_total'
 'cluster_node_count' 'cluster_number_of_pod' 'cluster_failed_node_count'
 'cluster_number_of_not_ready_pods' 'apiserver_non_2xx_read_request_qps'
 'apiserver_non_2xx_write_request_qps' 'apiserver_concurrent_connections'
 'apiserver_requests_per_second' 'namespace_number_of_not_ready_pods'
 'namespace_number_of_pods' 'error_count' 'slow_count' 'request_count'
 'avg_request_latency_seconds' 'arms_jvm_gc_total'
 'arms_jvm_gc_seconds_total' 'arms_jvm_gc_delta'
 'arms_jvm_gc_seconds_delta' 'arms_jvm_mem_used_bytes'
 'arms_jvm_mem_init_bytes' 'arms_jvm_mem_committed_bytes'
 'arms_jvm_mem_max_bytes' 'arms_jvm_threads_count'
 'arms_thread_pool_active_thread_count'
 'arms_thread_pool_current_thread_count'
 'arms_thread_pool_max_thread_count' 'arms_thread_pool_max_pool_size'
 'arms_thread_pool_core_pool_size' 'arms_thread_pool_scheduled_task_count'
 'arms_thread_pool_completed_task_count'
 'arms_thread_pool_rejected_task_count' 'arms_thread_pool_queue_size'
 'arms_grouped_thread_count' 'arms_grouped_thread_cpu_time'
 'arms_grouped_thread_runnable_count' 'arms_grouped_thread_waiting_count'
 'arms_grouped_thread_timed_waiting_count' 'cpu' 'latency' 'mem'
 'workload' 'error' 'slow' 'error_rate']```
```

In [ ]:
# In[3]:

# Ensure 'time' and 'value' columns are numeric
metrics_df['time'] = pd.to_numeric(metrics_df['time'], errors='coerce')
metrics_df['value'] = pd.to_numeric(metrics_df['value'], errors='coerce')

# Filter for the 'latency' metric in the 'apm.operation' entity set
latency_data = metrics_df[(metrics_df['metric'] == 'latency') & (metrics_df['entity_set'] == 'apm.operation')]

# Check if latency_data is not empty before calculating the P95 threshold
if not latency_data.empty:
    # Calculate the global P95 threshold for the 'latency' metric
    p95_threshold = np.percentile(latency_data['value'], 95)
else:
    p95_threshold = None

# Convert alert time window to timestamps in microseconds
timezone = pytz.timezone('Asia/Shanghai')
start_time = int(datetime.fromisoformat(alert_window['start']).astimezone(timezone).timestamp() * 1_000_000)
end_time = int(datetime.fromisoformat(alert_window['end']).astimezone(timezone).timestamp() * 1_000_000)

# Filter data for the alert time window and the specific operation
alert_latency_data = latency_data[
    (latency_data['time'] >= start_time) &
    (latency_data['time'] <= end_time) &
    (latency_data['entity_name'] == alert_entity['entity_name'])
]

# Check if alert_latency_data is not empty before identifying anomalies
if not alert_latency_data.empty and p95_threshold is not None:
    # Identify anomalies where latency exceeds the P95 threshold
    anomalies = alert_latency_data[alert_latency_data['value'] > p95_threshold]
else:
    anomalies = pd.DataFrame()  # Empty DataFrame if no data or no threshold

p95_threshold, anomalies

```
Out[3]:
```
The global P95 threshold for the `latency` metric could not be calculated because the relevant data was missing or empty. Additionally, no anomalies in the `latency` metric for the `cart::POST /oteldemo.CartService/AddItem` operation were identified within the alert time window.

The original code execution output of IPython Kernel is also provided below for reference:

(None, Empty DataFrame
Columns: []
Index: [])```
```

In [ ]:
# In[4]:

# Define the metrics to analyze
metrics_to_analyze = ['error_count', 'slow_count', 'request_count', 'error_rate']

# Initialize a dictionary to store global P95 thresholds
p95_thresholds = {}

# Calculate global P95 thresholds for each metric
for metric in metrics_to_analyze:
    metric_data = metrics_df[(metrics_df['metric'] == metric) & (metrics_df['entity_set'] == 'apm.operation')]
    if not metric_data.empty:
        p95_thresholds[metric] = np.percentile(metric_data['value'], 95)
    else:
        p95_thresholds[metric] = None

# Filter data for the alert time window and the specific operation
alert_metrics_data = metrics_df[
    (metrics_df['time'] >= start_time) &
    (metrics_df['time'] <= end_time) &
    (metrics_df['entity_name'] == alert_entity['entity_name']) &
    (metrics_df['metric'].isin(metrics_to_analyze))
]

# Identify anomalies for each metric
if not alert_metrics_data.empty:
    alert_metrics_data['anomaly'] = alert_metrics_data.apply(
        lambda row: row['value'] > p95_thresholds[row['metric']] if p95_thresholds[row['metric']] is not None else False,
        axis=1
    )
    anomalies = alert_metrics_data[alert_metrics_data['anomaly']]
else:
    anomalies = pd.DataFrame()  # Empty DataFrame if no data

p95_thresholds, anomalies

```
Out[4]:
```
The global P95 thresholds for the metrics are as follows: `error_count` = 54,910.95, `slow_count` = 1,553.4, `request_count` = 11,233.85, and `error_rate` = 0.98. However, no anomalies in these metrics were identified for the `cart::POST /oteldemo.CartService/AddItem` operation within the alert time window.

The original code execution output of IPython Kernel is also provided below for reference:

({'error_count': np.float64(54910.95), 'slow_count': np.float64(1553.4000000000012), 'request_count': np.float64(11233.849999999999), 'error_rate': np.float64(0.9801068754758461)}, Empty DataFrame
Columns: [time, domain, entity_set, entity_id, entity_name, metric, value, metric_set_id, service, anomaly]
Index: [])```
```

In [ ]:
# In[5]:

# Load the traces.parquet file
traces_file_path = 'rca100/cases/t086/traces.parquet'
traces_df = pd.read_parquet(traces_file_path)

# Ensure 'startTime', 'endTime', 'duration', and 'statusCode' columns are numeric
traces_df['startTime'] = pd.to_numeric(traces_df['startTime'], errors='coerce')
traces_df['endTime'] = pd.to_numeric(traces_df['endTime'], errors='coerce')
traces_df['duration'] = pd.to_numeric(traces_df['duration'], errors='coerce')
traces_df['statusCode'] = pd.to_numeric(traces_df['statusCode'], errors='coerce')

# Filter data for the alert time window
alert_traces_data = traces_df[
    (traces_df['startTime'] >= start_time * 1_000) &  # Convert microseconds to nanoseconds
    (traces_df['endTime'] <= end_time * 1_000) &  # Convert microseconds to nanoseconds
    (traces_df['spanName'] == alert_entity['entity_name'])
]

# Identify slow traces (duration above the 95th percentile globally)
if not traces_df.empty:
    global_p95_duration = np.percentile(traces_df['duration'], 95)
    slow_traces = alert_traces_data[alert_traces_data['duration'] > global_p95_duration]
else:
    slow_traces = pd.DataFrame()

# Identify error traces (statusCode != 0)
error_traces = alert_traces_data[alert_traces_data['statusCode'] != 0]

slow_traces, error_traces

```
Out[5]:
```
No slow or error traces were identified for the `cart::POST /oteldemo.CartService/AddItem` operation within the alert time window.

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [traceId, spanId, parentSpanId, kind, spanName, startTime, endTime, duration, serviceName, pid, hostname, statusCode, statusMessage, traceState, resources, attributes, __topic__, __tag__:__pack_id__, events, ip, links]
Index: []

[0 rows x 21 columns], Empty DataFrame
Columns: [traceId, spanId, parentSpanId, kind, spanName, startTime, endTime, duration, serviceName, pid, hostname, statusCode, statusMessage, traceState, resources, attributes, __topic__, __tag__:__pack_id__, events, ip, links]
Index: []

[0 rows x 21 columns])```
```

In [ ]:
# In[6]:

# Load the logs.parquet file
logs_file_path = 'rca100/cases/t086/logs.parquet'
logs_df = pd.read_parquet(logs_file_path)

# Ensure '_time_' column is numeric
logs_df['_time_'] = pd.to_numeric(logs_df['_time_'], errors='coerce')

# Filter logs for the alert time window and the 'cart' service
alert_logs = logs_df[
    (logs_df['_time_'] >= start_time) &
    (logs_df['_time_'] <= end_time) &
    (logs_df['_container_name_'].str.contains('cart', case=False, na=False))
]

# Search for logs containing error messages or unusual patterns
error_logs = alert_logs[alert_logs['content'].str.contains('error|exception|fail|critical', case=False, na=False)]

error_logs

```
Out[6]:
```
No error logs or unusual patterns were found for the `cart` service within the alert time window.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [content, _time_, _source_, _pod_uid_, _container_ip_, _image_name_, _container_name_, _pod_name_, _namespace_, __topic__, __tag__:__pack_id__, __tag__:__hostname__, __tag__:_node_name_, __tag__:_node_ip_, __tag__:_cluster_id_]
Index: []

[0 rows x 15 columns]```
```

In [ ]:
# In[7]:

# Load the topology.json file
topology_file_path = 'rca100/cases/t086/topology.json'

with open(topology_file_path, 'r') as file:
    topology_data = json.load(file)

# Extract entities and edges from the topology data
entities = topology_data.get('entities', [])
edges = topology_data.get('edges', [])

# Identify upstream services or nodes related to the 'cart' service
cart_service_id = next((entity['id'] for entity in entities if entity['name'] == 'cart'), None)

if cart_service_id:
    upstream_relations = [
        edge for edge in edges
        if edge['dst'] == cart_service_id and edge['relation'] == 'calls'
    ]
else:
    upstream_relations = []

upstream_relations

```
Out[7]:
```
The service dependency graph shows that two upstream services have a `calls` relation to the `cart` service. These services are potential sources of issues affecting the `cart` service. Their IDs are `7b50436b7fb9e828f0a8c32a69c53ab0` and `55e5d48e4d1bdc8b07f350cb89476ad1`.

The original code execution output of IPython Kernel is also provided below for reference:

[{'src': '7b50436b7fb9e828f0a8c32a69c53ab0', 'src_type': 'apm.service', 'dst': '469f8e313055adba13ca3f4e76c65505', 'dst_type': 'apm.service', 'relation': 'calls', 'first_observed': 0, 'last_observed': 1775237833}, {'src': '55e5d48e4d1bdc8b07f350cb89476ad1', 'src_type': 'apm.service', 'dst': '469f8e313055adba13ca3f4e76c65505', 'dst_type': 'apm.service', 'relation': 'calls', 'first_observed': 0, 'last_observed': 1775237833}]```
```

In [ ]:
# In[8]:

# Define the upstream service IDs
upstream_service_ids = ['7b50436b7fb9e828f0a8c32a69c53ab0', '55e5d48e4d1bdc8b07f350cb89476ad1']

# Define the metrics to analyze
metrics_to_analyze_upstream = ['latency', 'error_count', 'slow_count', 'request_count', 'error_rate']

# Initialize a dictionary to store global P95 thresholds for upstream services
upstream_p95_thresholds = {}

# Calculate global P95 thresholds for each metric for the upstream services
for metric in metrics_to_analyze_upstream:
    metric_data_upstream = metrics_df[
        (metrics_df['metric'] == metric) & 
        (metrics_df['entity_set'] == 'apm.service') & 
        (metrics_df['entity_id'].isin(upstream_service_ids))
    ]
    if not metric_data_upstream.empty:
        upstream_p95_thresholds[metric] = np.percentile(metric_data_upstream['value'], 95)
    else:
        upstream_p95_thresholds[metric] = None

# Filter data for the alert time window and the upstream services
alert_upstream_metrics_data = metrics_df[
    (metrics_df['time'] >= start_time) &
    (metrics_df['time'] <= end_time) &
    (metrics_df['entity_id'].isin(upstream_service_ids)) &
    (metrics_df['metric'].isin(metrics_to_analyze_upstream))
]

# Identify anomalies for each metric for the upstream services
if not alert_upstream_metrics_data.empty:
    alert_upstream_metrics_data['anomaly'] = alert_upstream_metrics_data.apply(
        lambda row: row['value'] > upstream_p95_thresholds[row['metric']] if upstream_p95_thresholds[row['metric']] is not None else False,
        axis=1
    )
    upstream_anomalies = alert_upstream_metrics_data[alert_upstream_metrics_data['anomaly']]
else:
    upstream_anomalies = pd.DataFrame()  # Empty DataFrame if no data

upstream_p95_thresholds, upstream_anomalies

```
Out[8]:
```
No global P95 thresholds could be calculated for the `latency`, `error_count`, `slow_count`, `request_count`, and `error_rate` metrics for the two upstream services. Additionally, no anomalies were identified in these metrics for the upstream services within the alert time window.

The original code execution output of IPython Kernel is also provided below for reference:

({'latency': None, 'error_count': None, 'slow_count': None, 'request_count': None, 'error_rate': None}, Empty DataFrame
Columns: []
Index: [])```
```

In [ ]:
# In[9]:

# Define the metrics to analyze for nodes
node_metrics_to_analyze = ['node_cpu_usage_rate', 'node_memory_usage_rate']

# Initialize a dictionary to store global P95 thresholds for node metrics
node_p95_thresholds = {}

# Calculate global P95 thresholds for each metric for all nodes
for metric in node_metrics_to_analyze:
    node_metric_data = metrics_df[
        (metrics_df['metric'] == metric) & 
        (metrics_df['entity_set'] == 'k8s.node')
    ]
    if not node_metric_data.empty:
        node_p95_thresholds[metric] = np.percentile(node_metric_data['value'], 95)
    else:
        node_p95_thresholds[metric] = None

# Identify the nodes hosting the 'cart' service from the topology data
cart_pod_ids = [
    edge['src'] for edge in edges
    if edge['dst'] == cart_service_id and edge['relation'] == 'hosts'
]
cart_node_ids = [
    edge['src'] for edge in edges
    if edge['dst'] in cart_pod_ids and edge['relation'] == 'hosts'
]

# Filter data for the alert time window and the nodes hosting the 'cart' service
alert_node_metrics_data = metrics_df[
    (metrics_df['time'] >= start_time) &
    (metrics_df['time'] <= end_time) &
    (metrics_df['entity_id'].isin(cart_node_ids)) &
    (metrics_df['metric'].isin(node_metrics_to_analyze))
]

# Identify anomalies for each metric for the nodes hosting the 'cart' service
if not alert_node_metrics_data.empty:
    alert_node_metrics_data['anomaly'] = alert_node_metrics_data.apply(
        lambda row: row['value'] > node_p95_thresholds[row['metric']] if node_p95_thresholds[row['metric']] is not None else False,
        axis=1
    )
    node_anomalies = alert_node_metrics_data[alert_node_metrics_data['anomaly']]
else:
    node_anomalies = pd.DataFrame()  # Empty DataFrame if no data

node_p95_thresholds, node_anomalies

```
Out[9]:
```
The global P95 thresholds for node metrics are as follows: `node_cpu_usage_rate` = 10.58% and `node_memory_usage_rate` = 91.45%. However, no anomalies were identified in these metrics for the nodes hosting the `cart` service within the alert time window.

The original code execution output of IPython Kernel is also provided below for reference:

({'node_cpu_usage_rate': np.float64(10.57671121970257), 'node_memory_usage_rate': np.float64(91.45467638026933)}, Empty DataFrame
Columns: []
Index: [])```
```

In [ ]:
# In[10]:

# Define the metrics to analyze for deployments
deployment_metrics_to_analyze = [
    'deployment_cpu_usage_total', 
    'deployment_memory_usage_total', 
    'deployment_availability_rate'
]

# Initialize a dictionary to store global P95 thresholds for deployment metrics
deployment_p95_thresholds = {}

# Calculate global P95 thresholds for each metric for all deployments
for metric in deployment_metrics_to_analyze:
    deployment_metric_data = metrics_df[
        (metrics_df['metric'] == metric) & 
        (metrics_df['entity_set'] == 'k8s.deployment')
    ]
    if not deployment_metric_data.empty:
        deployment_p95_thresholds[metric] = np.percentile(deployment_metric_data['value'], 95)
    else:
        deployment_p95_thresholds[metric] = None

# Identify the deployment associated with the 'cart' service from the topology data
cart_deployment_ids = [
    edge['src'] for edge in edges
    if edge['dst'] == cart_service_id and edge['relation'] == 'contains'
]

# Filter data for the alert time window and the deployment associated with the 'cart' service
alert_deployment_metrics_data = metrics_df[
    (metrics_df['time'] >= start_time) &
    (metrics_df['time'] <= end_time) &
    (metrics_df['entity_id'].isin(cart_deployment_ids)) &
    (metrics_df['metric'].isin(deployment_metrics_to_analyze))
]

# Identify anomalies for each metric for the deployment associated with the 'cart' service
if not alert_deployment_metrics_data.empty:
    alert_deployment_metrics_data['anomaly'] = alert_deployment_metrics_data.apply(
        lambda row: row['value'] > deployment_p95_thresholds[row['metric']] if deployment_p95_thresholds[row['metric']] is not None else False,
        axis=1
    )
    deployment_anomalies = alert_deployment_metrics_data[alert_deployment_metrics_data['anomaly']]
else:
    deployment_anomalies = pd.DataFrame()  # Empty DataFrame if no data

deployment_p95_thresholds, deployment_anomalies

```
Out[10]:
```
The global P95 thresholds for deployment metrics are as follows: `deployment_cpu_usage_total` = 0.39, `deployment_memory_usage_total` = 2,031,450,521.6 bytes, and `deployment_availability_rate` could not be calculated (NaN). However, no anomalies were identified in these metrics for the deployment associated with the `cart` service within the alert time window.

The original code execution output of IPython Kernel is also provided below for reference:

({'deployment_cpu_usage_total': np.float64(0.39102765328056394), 'deployment_memory_usage_total': np.float64(2031450521.6000004), 'deployment_availability_rate': np.float64(nan)}, Empty DataFrame
Columns: []
Index: [])```
```

In [ ]:
# In[11]:

# Inspect the columns of the events DataFrame to identify the correct timestamp column
events_columns = events_df.columns

# Display the columns to identify the correct timestamp column
events_columns

```
Out[11]:
```
The `events.parquet` file contains columns such as `eventId`, `hostname`, `level`, `clusterId`, `clusterName`, `pod_id`, and `pod_name`. However, it does not include a clear timestamp column to filter events by the alert time window. Further investigation is needed to determine how to filter events based on time.

The original code execution output of IPython Kernel is also provided below for reference:

Index(['eventId', 'hostname', 'level', 'clusterId', 'clusterName', '__topic__', 'pod_id', 'pod_name'], dtype='object')```
```

In [ ]:
# In[12]:

# Parse the `eventId` field to extract detailed event information
# Since the error indicates that `eventId` is already a dictionary, we can directly access its keys
events_df['reason'] = events_df['eventId'].apply(lambda x: x.get('reason') if isinstance(x, dict) else None)
events_df['message'] = events_df['eventId'].apply(lambda x: x.get('message') if isinstance(x, dict) else None)
events_df['involvedObject'] = events_df['eventId'].apply(lambda x: x.get('involvedObject') if isinstance(x, dict) else None)

# Filter events related to the `cart` service or its associated pods, nodes, or deployments
cart_related_events = events_df[
    events_df['message'].str.contains('cart', case=False, na=False) |
    events_df['involvedObject'].apply(lambda x: 'cart' in str(x).lower() if x else False)
]

cart_related_events

```
Out[12]:
```
No Kubernetes events related to the `cart` service, its associated pods, nodes, or deployments were found within the available data.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [eventId, hostname, level, clusterId, clusterName, __topic__, pod_id, pod_name, reason, message, involvedObject]
Index: []

[0 rows x 11 columns]```
```